# 07 · Train the student (E0 baselines)

Trains the student on road-labelled KITTI-360 frames: every config in `CONFIGS`, one after another in the same session (the frames are staged once, and configs that build the same BEV records share them), each into its own output folder `dev_<config>_s<seed>`:

| Config | Baseline |
|---|---|
| `configs/student/e0_camera_ipm.yaml` | E0a camera only, flat-plane IPM from the calibration, no LiDAR anywhere |
| `configs/student/e0_lidar.yaml` | E0b LiDAR only |
| `configs/student/e0_concat.yaml` | E0c simple fusion: camera lifted onto the scan's ground + LiDAR, concatenated |
| `configs/student/e1_posthoc.yaml` | E1 baseline arm: simple fusion, box-aware occupancy targets; boxes subtracted after prediction (notebook 08, `POST_HOC`) |
| `configs/student/e1_objects.yaml` | E1 object arm (= E2 concat arm): simple fusion + object channels, boxes as occluders, box-aware targets |
| `configs/student/e2_gate.yaml` | E2 gate arm (OCGF): camera and LiDAR weighed per cell by the learned gate, from Q and the object channels |
| `configs/student/e4_fixed.yaml`, `e4_gate_noq.yaml`, `e4_gate_q.yaml` | E4: fixed 0.5/0.5 fusion; learned gate without Q; with Q (no objects) |
| `configs/student/s3_no_consistency.yaml` | S3: the main model (E2 gate) without the self-supervised consistency terms |
| `configs/student/e3_det2d_mtrans.yaml`, `e3_oracle2d_mtrans.yaml`, `e3_gt3d.yaml` | E3 on the main model: objects from MTrans on the detector's boxes (real pipeline); **ORACLE** MTrans on ground-truth 2D boxes; **ORACLE** ground-truth 3D boxes (notebook 04 builds the oracle files) |
| `configs/student/a1_lidar_anchored.yaml`, `a2_flat_plane.yaml`, `a3_vis_lidar_only.yaml` | Lifting ablations on the main model: camera sampled at LiDAR points; at the flat plane; visibility without box occluders |

Every config shares the loss of `base.yaml`: road, occupancy, unknown, auxiliary heads, road boundary, and the self-supervised cross-modal and temporal consistency terms (the temporal term pairs each frame with the one about 5 m earlier).

Inputs per frame: the camera image and the LiDAR scan (and, for the ground-guided camera lifting, the ground surface estimated from that scan). Targets: the road teacher's labels lifted to the BEV grid (`ocgf/training/targets.py`), built at the start of each session. Validation loss is measured against the teachers' labels: it only shows convergence and is **never a reported metric** (metrics come from the evaluation ground truth, notebook 08).

Starter phase: drive 0000 with the frame-level split `kitti360_d0000_dev_v1` (debugging, never paper results).

## Before you run

1. **Settings → Accelerator:** GPU T4. **Internet:** On (code clone, DINOv2 ViT-S weights, about 90 MB).
2. **Add Input:** drive 0000's data-prep output (the full run; the inputs cell flags a trial), the KITTI-360 starter (images, calibration, poses) and **`ocgf-road-labels-v1`** (notebook 05). For the E1 configs also **`ocgf-mtrans-objects-v1`** (notebook 04).
3. To continue a stopped run (12 h limit), also attach **this notebook's previous output**; the run resumes from its checkpoint.
4. **Save Version → Save & Run All.**

In [ ]:
from pathlib import Path
import time

T0 = time.time()                     # the time budget is measured from here
REPO_URL   = "https://github.com/GNaveenkumar614/OCGF.git"
GIT_REF    = "p0/skeleton-geometry"  # branch, tag or commit (main once merged); pin a commit for the final run
CONFIGS    = ["configs/student/e0_concat.yaml",      # trained in turn, each into its own output folder
              "configs/student/e0_lidar.yaml",
              "configs/student/e0_camera_ipm.yaml"]
SEED       = 0
EXPERIMENTS = {c: f"dev_{Path(c).stem}_s{SEED}" for c in CONFIGS}   # output folders; keep them to resume
SPLIT      = "configs/splits/kitti360_d0000_dev_v1.json"
DRIVES     = ["2013_05_28_drive_0000_sync"]
EPOCHS     = None                    # None: the config's
BACKBONE   = None                    # None: the config's (DINOv2 ViT-S/14)
RUN_SMOKE_TEST = True                # a few steps and a checkpoint reload before the real run
TIME_BUDGET_HOURS = 11.0             # stop cleanly this long after the notebook started (a session ends at 12 h)
THREADS    = 32                      # parallel readers: /kaggle/input is slow per file

INPUT_ROOT = Path("/kaggle/input")
WORKING    = Path("/kaggle/working")
STAGE_DIR  = Path("/kaggle/temp/stage")            # local copy of the frames' scans and images
RECORDS    = Path("/kaggle/temp/bev_records")      # BEV inputs and targets (one subfolder per kind), each session
CODE_DIR   = Path("/kaggle/temp/OCGF")

## Code

A fresh clone of the repo at `GIT_REF`, installed in editable mode.

In [ ]:
import os, shutil, subprocess, sys

def sh(cmd, cwd=None):
    # run a command, stream its output into this notebook, and stop Run All if it fails
    print(f"$ {cmd}", flush=True)
    proc = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                            env=dict(os.environ, PYTHONUNBUFFERED="1"))
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait():
        raise RuntimeError(f"command failed with exit code {proc.returncode}: {cmd}")

CODE_DIR.parent.mkdir(parents=True, exist_ok=True)
if CODE_DIR.exists():
    shutil.rmtree(CODE_DIR)            # fresh clone, so the run matches GIT_REF exactly
sh(f'git clone -q {REPO_URL} "{CODE_DIR}"')
sh(f'git -C "{CODE_DIR}" checkout -q {GIT_REF}')
sh(f'git -C "{CODE_DIR}" log -1 --date=short --format="code at commit %h (%cd): %s"')
sh(f'{sys.executable} -m pip install -q -e "{CODE_DIR}"')
sys.path.insert(0, str(CODE_DIR))
for name in [m for m in sys.modules if m == "ocgf" or m.startswith("ocgf.")]:
    del sys.modules[name]

from ocgf.utils.provenance import environment
print(environment())
sh("nvidia-smi --query-gpu=name,memory.total --format=csv")

## Inputs and resume

Each drive needs its data-prep output, the starter and the road labels. A previous session's output (attached as input) is copied back so the run continues.

In [ ]:
import json
from ocgf.datasets.kitti360 import Kitti360Sequence
from ocgf.datasets.locate import describe_prepared, find_outputs, find_runs, prepared_drive

label_roots = set()
for seq in DRIVES:
    parts = prepared_drive(INPUT_ROOT, seq)          # raises if the drive's inputs are not attached
    print(describe_prepared(INPUT_ROOT, seq))
    labels = find_outputs(INPUT_ROOT, seq, "teacher", "road")
    assert labels, f"no road labels for {seq}: attach ocgf-road-labels-v1 (Add Input)"
    n = len(list(labels[0].glob("*.npz")))
    expected = len(Kitti360Sequence(sequence=seq, components=parts).usable_frames(sampled=True))
    print(f"{seq}: scans {parts['scans']}; road labels {labels[0]} ({n} of {expected} frames sampled every 5 m)")
    if n < 0.95 * expected:
        print(f"  WARNING: the road labels cover only {n} of {expected} frames: a trial run of notebook 05? "
              f"Training will use only those frames.")
    label_roots.add(labels[0].parent)
assert len(label_roots) == 1, "keep every drive's road labels in one dataset"
ROAD_LABELS = label_roots.pop()

def epochs_done(ckpt_dir):
    p = ckpt_dir / "progress.json"
    return json.loads(p.read_text())["epoch"] + 1 if p.exists() else 0

from ocgf.training.experiment import load_config, resolve

def records_kind(config):
    # configs that build the same BEV records share a folder (E0 configs all build the plain ones)
    r = resolve(load_config(CODE_DIR / config))
    return f"occluders{int(r.inputs.box_occluders)}_targets{int(r.targets.box_targets)}_{r.object_source}"

def args(config):
    return (f'--config "{CODE_DIR / config}" --input-root "{INPUT_ROOT}" --sequence {" ".join(DRIVES)} '
            f'--road-labels "{ROAD_LABELS}" --objects "{INPUT_ROOT}" --split "{CODE_DIR / SPLIT}" '
            f'--records "{RECORDS / records_kind(config)}" --stage-dir "{STAGE_DIR}" --threads {THREADS} --seed {SEED}'
            + (f" --epochs {EPOCHS}" if EPOCHS else "") + (f" --backbone {BACKBONE}" if BACKBONE else ""))

for config, name in EXPERIMENTS.items():
    out = WORKING / name
    if not (out / "ckpt" / "last.pt").exists():
        previous = find_runs(INPUT_ROOT, name)
        if previous:
            source = max(previous, key=lambda d: epochs_done(d / "ckpt"))
            shutil.copytree(source, out, dirs_exist_ok=True)
            print(f"{name}: continuing from {source}")
    print(f"{name}: " + (f"{epochs_done(out / 'ckpt')} epochs done" if (out / "ckpt").exists() else "a fresh run")
          + f"; records {records_kind(config)}")

## Smoke test

A few steps of the first config and a checkpoint reload, into a scratch folder (this also stages the frames and builds the BEV records the real runs use).

In [ ]:
if RUN_SMOKE_TEST:
    sh(f'{sys.executable} scripts/train_student.py {args(CONFIGS[0])} --out /kaggle/temp/smoke --smoke 4', cwd=CODE_DIR)

## Train

The configs in turn. Each resumes from its own `ckpt/last.pt` if it exists (a finished run is skipped quickly) and stops cleanly before the time budget; run again with this output attached to continue. Progress also goes to each folder's `train.log`.

In [ ]:
for config, name in EXPERIMENTS.items():
    remaining = TIME_BUDGET_HOURS - (time.time() - T0) / 3600
    if remaining < 0.3:
        print(f"{name}: skipped, the time budget is used up; run again with this output attached")
        continue
    sh(f'{sys.executable} scripts/train_student.py {args(config)} --out "{WORKING / name}" '
       f'--time-budget-hours {remaining:.2f}', cwd=CODE_DIR)

## Summary

Per run: epochs done and the training loss per epoch (and the validation loss against the teachers' labels, a convergence check only).

In [ ]:
for config, name in EXPERIMENTS.items():
    out = WORKING / name
    if not (out / "ckpt" / "progress.json").exists():
        print(f"{name}: no checkpoint yet")
        continue
    progress = json.loads((out / "ckpt" / "progress.json").read_text())
    print(f"\n{name}: {progress['epoch'] + 1} of {progress['epochs']} epochs done"
          + ("" if progress["epoch"] + 1 >= progress["epochs"] else "; attach this output and run again to continue"))
    for e in map(json.loads, (out / "metrics.jsonl").read_text().splitlines()):
        val = e.get("val_loss_vs_teacher", {})
        print(f"  epoch {e['epoch']:3d}  train loss {e['train']['loss']:.4f} (road {e['train']['road']:.4f}, "
              f"occ {e['train']['occ']:.4f}, unk {e['train']['unk']:.4f})"
              + (f"  | val vs teacher {val['loss']:.4f}" if val else "") + f"  {e['seconds']:.0f} s")
sh(f'du -sh "{WORKING}"/dev_*')

## After the run

The output holds one folder per config, `dev_<config>_s<seed>` (checkpoint, `metrics.jsonl`, `train.log`, `manifest.json` with the resolved config, seed and git commit). Save it as one dataset (Output → New Dataset), e.g. **`ocgf-runs-e0-s0`**; notebook 08 finds every run by its folder name. Evaluation against the ground truth comes in notebook 08.